Logistic Regression is a supervised ML model analogues to Linear Regression, but tries to predict for categorical variables or discrete ones (yes/no), to be more precise Logistic Regression is a variation of Linear Regression.

When it is suitable?

- if data is binary 0/1, yes/no, True/False
- if you need probabilistic results.

### 1 . Hypothetical function :
---

The output from the hypothesis is the estimated probability. This is used to infer how confident can predicted value be actual value when given an input X. 

$$\hat{Y} = sigmoid(b + \theta_1x_1 + \theta_2x_2 + ... + \theta_nx_n) = P(Y=1|X) $$

Where
$$ sigmoid(t)=\frac{1}{1+e^{-t}}$$

<img src="https://miro.medium.com/max/700/1*RqXFpiNGwdiKBWyLJc_E7g.png
" width="400"></img>

 $b$ is the biais, and $ \theta_i $ are the weights of the linear model.


### 2 . Cost function :
---

We need now to set a specific function to calculate the error of our model since it is not perfect, we call this function $cost function$ :

<blockquote> Cost function is an error function which is used to find the error term of a model on a data.
    With $\hat{y}_i$ is the model’s output label.  
</blockquote>
    

$$J(\theta) = \frac{1}{m}\sum \limits _{i=1} ^{m} \hat{y}_i log(\hat{y}_i) + (1-\hat{y}_i)log(1-\hat{y}_i)$$

For just two variables with $\theta_0$ is our biais b, we get the shape in the picture :

<img src="https://www.researchgate.net/profile/Alp-Yilmaz/publication/342720152/figure/fig3/AS:910374819864576@1594061679394/Cost-function-for-Logistic-Regression.png" 
width="350"></img>

##### Why cost function which has been used for linear can not be used for logistic?

Linear regression uses mean squared error as its cost function. If this is used for logistic regression, then it will be a non-convex function of parameters (theta). Gradient descent will converge into global minimum only if the function is convex.


### 3 . Gradient descent and updates :
---

Can you set directly the parameters of our Hypothetical function and get the precise predictions you are looking for?! Obviously No even with a little chance! 

If we change randomly the values we may pick the best ones but in the same time our chance to be lost is too large!

Indeed, the only efficient way is to get these parameters when the cost function is in its minimum.

and that is the concept of the famous method Gradient Descent.

<blockquote> Gradient Descent algorithm will change, at each iteration, the values of all $\theta_i$ and the biais $b$ until the best possible combinaition of parameters is found. </blockquote>
    
   
Imagine that you are on a hill, and you want to go down it. With each new step (analogy to iteration), you look around to find the best slope to move down. Once you have found the slope, you take a step forward one magnitude $\alpha$.

Here is an example with one variable how it looks, with w is our parameter in that case :


<img src="https://cdn-images-1.medium.com/max/1600/0*fU8XFt-NCMZGAWND." width="400"></img>


<br>

Here are the formulas of Gradient Descent for the hypothesis function we are working on from the beginning :

$$ \theta_i = \theta_i - \alpha d\theta_i$$

$$ b = b - \alpha db$$

Here are the derivatives we need, you can replace them in the equations :

$$d \theta_i = - \frac{2}{m} \sum \limits _{i=1} ^{m} x_i (\hat{y_i} - y)$$

$$d b = - \frac{2}{m}\sum \limits _{i=1} ^{m} (\hat{y_i} - y)$$


## Coding Time :

We choose to work with OOP since it is appropriate with that kind of problems, and all models are implemented using OOP in Sickit learn.

The two functions we need to code are fit and predict, the first one is for model training, i.e it will find the parameters of our model based on the number of iterations and learning rate we are working with; and the second one to get the predictions for new samples.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [2]:
class LogisticRegression :
    def __init__(self, iterations , lr):
        self.iterations = iterations
        self.lr = lr
        self.b = None
        
    def fit(self , x , y):
        self.x = x
        self.y = y
        self.m , self.n = x.shape
        self.weights = np.zeros(self.n)
        self.b = 0
        
        #Implementing Gradient Descent
        for i in range(self.iterations):
            linReg = np.dot(self.x, self.weights) + self.b
            y_pred = self.sigmoid(linReg)
            
            #Calculating derivatives w.r.t Parameters
            D_w = (1/self.m)*np.dot(self.x.T, (y_pred - self.y)) # we could use * because they are arrays # that was the pb why it was not working!
            D_b = (1/self.m)*np.sum(y_pred - self.y)
            
            #Updating Parameters
            self.weights = self.weights - self.lr * D_w
            self.b = self.b - self.lr * D_b
            
    def sigmoid(self, X) :
        return 1 / (1 + np.exp(-X))
            
    def predict(self , x):
        linReg = np.dot(x, self.weights) + self.b
        y_preds = self.sigmoid(linReg)
        
        final_preds = [1 if pred>0.5 else 0 for pred in y_preds]
        return final_preds
        

Let's apply it on Titanic dataset.

For pre-processing part I just copied it from one of my previous kernels in this competition, you can take a look at it if you don't understand something.

Here is the link :

https://www.kaggle.com/khkuggle/simple-and-intermediate-eda-modeling-for-titanic

In [3]:
train = pd.read_csv('/kaggle/input/titanic/train.csv')
test = pd.read_csv('/kaggle/input/titanic/test.csv')

train.drop('PassengerId', axis = 1, inplace = True)
df = train.append(test).reset_index(drop=True)
for i in range(len(df)):
    if not(pd.isnull(df['Cabin'].iloc[i])):
        df['Cabin'].iloc[i]=df['Cabin'].iloc[i][0] 
    else :
        df['Cabin'].iloc[i]='No'
    

n = len(train)

df['Fsize'] = df['Parch'] + df['SibSp'] + 1
df['travelled_alone'] = 'No'
df.loc[df.Fsize == 1, 'travelled_alone'] = 'Yes'
df['Title'] = df['Name'].str.extract('([A-Za-z]+)\.', expand=False)
df['Title'] = df['Title'].replace(['Capt', 'Col', 'Rev', 'Don', 'Countess', 'Jonkheer', 'Dona', 'Sir', 'Dr', 'Major', 'Dr'], 'Rare')
df['Title'] = df['Title'].replace(['Mlle', 'Mme', 'Ms'], 'Miss')
df['Title'] = df['Title'].replace(['Lady'], 'Mrs')
df.Embarked.fillna(train.Embarked.mode()[0], inplace = True)
mean = train["Age"].mean()
std = train["Age"].std()

is_null = df["Age"].isnull().sum()
# compute random numbers between the mean, std and is_null
rand_age = np.random.randint(mean - std, mean + std, size = is_null)
# fill NaN values in Age column with random values generated
age_slice = df["Age"].copy()
age_slice[np.isnan(age_slice)] = rand_age

df["Age"] = age_slice
df["Age"] = df["Age"].astype(int)
df['Fare'].fillna(train['Fare'].mean(), inplace = True)

features = ["Sex", "Pclass","travelled_alone", "Cabin", "Embarked", "Title"]

df=pd.get_dummies(df,columns=features,drop_first=True)
df.drop(['Name', 'Ticket'], axis = 1, inplace = True)

train = df[:n ] # the three outliers
test = df[n:]

X = train.drop(['Survived','PassengerId'], axis = 1)
y = train.Survived
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

/opt/conda/lib/python3.7/site-packages/pandas/core/indexing.py:1732: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_single_block(indexer, value, name)


In [4]:
model = LogisticRegression(1000, 0.001)

In [5]:
model.fit(x_train,y_train)

In [6]:
pred = model.predict(x_test)

In [7]:
pred[10:20]

[0, 0, 0, 1, 1, 0, 0, 0, 0, 0]

In [8]:
y_test[10:20]

704    0.0
346    1.0
196    0.0
535    1.0
310    1.0
14     0.0
350    0.0
145    0.0
614    0.0
803    1.0
Name: Survived, dtype: float64

In [9]:
accuracy_score(y_test, pred)

0.7206703910614525

We got a good score.

Let's try now with the real model from scikit learn:

In [10]:
from sklearn.linear_model import LogisticRegression

In [11]:
lr = LogisticRegression(solver='liblinear')
lr.fit(x_train,y_train)

LogisticRegression(solver='liblinear')

In [12]:
pred2 = lr.predict(x_test)
accuracy_score(y_test, pred2)

0.8156424581005587

## If you like that kernel, please don't forget to upvote it !